# TimesFM 3 P0a: calibration-only covariate harm screen

Runs the same frozen P0a screen as the Chronos-2 notebook using the pinned official FEV TimesFM-3 wrapper. TimesFM 3 weights are used only for academic, non-commercial research under their upstream license.

Use an **A100 or H100** if available. A T4 is valid but slower. Completed task-variant units are saved directly to Google Drive and are resumable after a disconnect.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
FEV_CHECKOUT = Path('/content/fev-pinned')
FEV_COMMIT = '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
TIMESFM_COMMIT = '20191171b74f51bfead932b6b8d0c8f515e70f63'

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

if not FEV_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', '--filter=blob:none',
         'https://github.com/autogluon/fev.git', str(FEV_CHECKOUT)],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(FEV_CHECKOUT), 'checkout', FEV_COMMIT],
        check=True,
    )
else:
    observed = subprocess.check_output(
        ['git', '-C', str(FEV_CHECKOUT), 'rev-parse', 'HEAD'], text=True
    ).strip()
    assert observed == FEV_COMMIT, observed

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        f'git+https://github.com/autogluon/fev.git@{FEV_COMMIT}',
        ('timesfm[torch] @ git+https://github.com/google-research/'
         f'timesfm.git@{TIMESFM_COMMIT}'),
    ],
    check=True,
)

SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Pinned FEV checkout:', FEV_CHECKOUT)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())
print('covsafe import:', covsafe.__file__)

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'Select Runtime > Change runtime type > GPU, then restart.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests/p0a'
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Durable output root:', OUTPUT_ROOT)

In [ ]:
import json

from covsafe.p0a import EXPECTED_P0A_CONFIG_HASH
from covsafe.timesfm3_p0a import run_timesfm3_p0a

print('Frozen P0a config hash:', EXPECTED_P0A_CONFIG_HASH)
report = run_timesfm3_p0a(REPO, FEV_CHECKOUT, OUTPUT_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

## Return artifact

Send the final JSON report printed above. `HF_TOKEN` is optional for this public checkpoint; adding it as a Colab secret only improves Hub rate limits. On reconnect, run all cells again and completed units will print `RESUME`.